# Organized execution copy

The exact executed predecessor with historical outputs is in `archive/revision_executed/`. This execution copy has cleared outputs. No new GPU experiment was run during packaging. Install, restart, then run the configuration cell and remaining cells in order. Edit DATA_ROOT and RUN_DIR consistently across notebooks; use a new RUN_DIR for a new reproduction and the same one to resume it. See `docs/RUN_ORDER.md` and `docs/DATA_SETUP.md` for required files. The conversion/model/scoring calls and experimental settings are retained.

Keep Session A and Session B separate. RUN_DIR must match between them so B reads A outputs.


# IJEEE-26-0020 — YourTTS on the 40-sentence holdout (ASR-mediated path)

This notebook adds the last missing system to the holdout comparison, so the old Table I can be replaced.
It reproduces the path described in Section III-D: **Whisper-large transcribes the Joe source sentence, and YourTTS resynthesises that text with the Kathleen reference**.

It needs **two Colab sessions**, because Coqui TTS needs a newer `transformers` than the scoring setup:

* **Session A (cells A1–A3)** — generate and time the 40 YourTTS outputs. Saved to Drive.
* **Session B (cells B1–B5)** — score them with exactly the same code as `IJEEE_revision_runs_final.ipynb` (Cells 1, 2 and 5 are copied unchanged).

Runtime: T4 GPU. Expected time: about 15 min (A) + 15 min (B).
Files written to `IJEEE_VC_data/revision_runs_final/`: `E15_audio/`, `E15_yourtts_generation.csv`, `environment_yourtts.json`, `E15_yourtts_holdout40.csv`, `E15_reference_format.csv`.
**Send back:** the zip created by cell B5.

## SESSION A — generation and timing (fresh session, T4 GPU)

In [ ]:
# Cell A1 — INSTALL (then Runtime → Restart session, and continue with A2)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "coqui-tts[codec]==0.28.0", "six"], check=True)
print("✅ Installed coqui-tts 0.28.0 (with torchcodec). NOW: Runtime → Restart session, then run Cell A2.")

In [ ]:
# Runtime configuration — edit these two lines consistently in all notebooks.
import os
DATA_ROOT = "/content/drive/MyDrive/IJEEE_VC_data"
RUN_DIR = "/content/drive/MyDrive/IJEEE_VC_data/reproduction_runs/review_run_01"
# DATA_ROOT contains audio/models. RUN_DIR is a new output folder, never results/.
os.environ["IJEEE_DATA_ROOT"] = DATA_ROOT
os.environ["IJEEE_RUN_DIR"] = RUN_DIR


In [ ]:
# Cell A2 — PATHS, MODELS, WARM-UP

import os
import json
import time
import random
import glob

from google.colab import drive

drive.mount("/content/drive")

import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
import importlib.metadata as im

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
)
from TTS.api import TTS

BASE_DIR = os.environ.get("IJEEE_DATA_ROOT", "/content/drive/MyDrive/IJEEE_VC_data")

OUT = os.environ.get("IJEEE_RUN_DIR", os.path.join(BASE_DIR, "reproduction_runs", "review_run_01"))
os.makedirs(OUT, exist_ok=True)

AUD = os.path.join(OUT, "E15_audio")
os.makedirs(AUD, exist_ok=True)

TARGET_REF = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav",
)

HOLDOUT_META = os.path.join(
    BASE_DIR,
    "final_holdout_data_40",
    "holdout40_metadata.csv",
)

H40 = pd.read_csv(
    HOLDOUT_META,
    dtype={"original_id": str},
).sort_values("sample").reset_index(drop=True)

assert len(H40) == 40
assert set(H40["sample"]) == set(range(1, 41))
assert os.path.isfile(TARGET_REF), "❌ Kathleen reference missing"

for path in H40["prepared_wav"]:
    assert os.path.isfile(path), f"❌ Source file missing: {path}"

assert torch.cuda.is_available(), "❌ Select a T4 GPU runtime"

DEV = "cuda"

print("GPU:", torch.cuda.get_device_name(0), flush=True)


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ASR stage — Whisper-large-v3
ASR_ID = "openai/whisper-large-v3"

print("Loading Whisper-large-v3...", flush=True)

proc = AutoProcessor.from_pretrained(ASR_ID)

asr = AutoModelForSpeechSeq2Seq.from_pretrained(
    ASR_ID,
    dtype=torch.float16,
    use_safetensors=True,
).to(DEV).eval()

asr.config.forced_decoder_ids = None
asr.generation_config.forced_decoder_ids = None


@torch.inference_mode()
def run_asr(path):
    torch.cuda.synchronize()
    start = time.perf_counter()

    audio = librosa.load(
        path,
        sr=16000,
        mono=True,
    )[0].astype(np.float32)

    inputs = proc(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        return_attention_mask=True,
    )

    arguments = {
        "input_features": inputs.input_features.to(
            DEV,
            torch.float16,
        ),
        "language": "en",
        "task": "transcribe",
        "max_new_tokens": 128,
    }

    if getattr(inputs, "attention_mask", None) is not None:
        arguments["attention_mask"] = inputs.attention_mask.to(DEV)

    generated = asr.generate(**arguments)

    text = proc.batch_decode(
        generated,
        skip_special_tokens=True,
    )[0].strip()

    torch.cuda.synchronize()

    return text, time.perf_counter() - start


# TTS stage — YourTTS
YT_NAME = "tts_models/multilingual/multi-dataset/your_tts"

print("Loading YourTTS...", flush=True)

yt = TTS(
    YT_NAME,
    progress_bar=False,
).to(DEV)

# Align speaker-reference inputs with the GPU encoder.
speaker_manager = yt.synthesizer.tts_model.speaker_manager

assert speaker_manager.encoder is not None, (
    "❌ Speaker encoder is missing"
)

assert hasattr(speaker_manager, "use_cuda"), (
    "❌ Speaker manager device setting not found — stop and report"
)

speaker_manager.use_cuda = True
speaker_manager.encoder.to(DEV).eval()

encoder_device = next(
    speaker_manager.encoder.parameters()
).device

assert encoder_device.type == "cuda"

print(
    "Speaker encoder device:",
    encoder_device,
    flush=True,
)


def run_yourtts(text, out_path, seed):
    set_all_seeds(seed)

    torch.cuda.synchronize()
    start = time.perf_counter()

    yt.tts_to_file(
        text=text,
        speaker_wav=TARGET_REF,
        language="en",
        file_path=out_path,
        split_sentences=False,
    )

    torch.cuda.synchronize()

    return time.perf_counter() - start


# Two warm-up runs
WARM = "/content/warmup"
os.makedirs(WARM, exist_ok=True)

for repetition in range(1, 3):
    print(
        f"Warm-up {repetition}/2...",
        flush=True,
    )

    text, _ = run_asr(H40.iloc[0]["prepared_wav"])

    run_yourtts(
        text,
        os.path.join(WARM, "yt.wav"),
        2026,
    )

ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "coqui_tts": im.version("coqui-tts"),
    "transformers": im.version("transformers"),
    "tts_model": YT_NAME,
    "tts_model_licence": "CC BY-NC-ND 4.0",
    "asr_model": ASR_ID,
    "asr_precision": "fp16",
    "tts_precision": "fp32",
    "batch_size": 1,
    "warmups": 2,
    "timed_repeats": 3,
    "cuda_synchronize": True,
    "speaker_embedding_inside_timed_span": True,
    "speaker_encoder_use_cuda": True,
    "device_fix": (
        "Explicitly enabled speaker_manager.use_cuda "
        "and moved its encoder to CUDA"
    ),
    "split_sentences": False,
    "tts_seed": "202900 + sample",
}

with open(
    os.path.join(OUT, "environment_yourtts.json"),
    "w",
    encoding="utf-8",
) as file:
    json.dump(ENV, file, indent=2)

print("✅ Ready")
print(ENV)

In [ ]:
# Cell A3 — 40 SENTENCES: 3 timed runs per stage + 1 end-to-end run (resumes if interrupted)
csv = os.path.join(OUT, "E15_yourtts_generation.csv")
done = set(pd.read_csv(csv)["sample"]) if os.path.exists(csv) else set()
for _, r in H40.iterrows():
    i = int(r["sample"])
    if i in done: continue
    src, out = r.prepared_wav, os.path.join(AUD, f"yourtts_s{i:02d}.wav")
    asr_runs = [run_asr(src) for _ in range(3)]
    text = asr_runs[-1][0]; t_asr = [t for _, t in asr_runs]
    t_tts = [run_yourtts(text, out, 202900 + i) for _ in range(3)]          # the last run's file is kept
    torch.cuda.synchronize(); t0 = time.perf_counter()                      # one end-to-end run: ASR + TTS in one span
    t_e2e_text, _ = run_asr(src); run_yourtts(t_e2e_text, out.replace(".wav", "_e2e.wav"), 202900 + i)
    torch.cuda.synchronize(); e2e = time.perf_counter() - t0
    row = dict(system="YourTTS (ASR-mediated)", sample=i, src_duration=r.duration_seconds, asr_transcript=text,
               t_asr_run1=t_asr[0], t_asr_run2=t_asr[1], t_asr_run3=t_asr[2],
               t_tts_run1=t_tts[0], t_tts_run2=t_tts[1], t_tts_run3=t_tts[2],
               t_asr=float(np.mean(t_asr)), t_tts=float(np.mean(t_tts)))
    row["latency"] = row["t_asr"] + row["t_tts"]; row["latency_e2e"] = e2e; row["rtf"] = row["latency"] / r.duration_seconds
    row["out_sr"] = sf.info(out).samplerate; row["output_file"] = os.path.basename(out)
    pd.DataFrame([row]).to_csv(csv, mode="a", header=not os.path.exists(csv), index=False)
    print(i, f"{row['latency']:.2f}s", text)
g = pd.read_csv(csv)
print(len(g), "sentences | mean latency", round(g.latency.mean(), 3), "s | ASR", round(g.t_asr.mean(), 3), "| TTS", round(g.t_tts.mean(), 3))
assert len(g) == 40, "❌ not all 40 sentences done — run this cell again"
print("✅ Session A finished. Open a NEW session (Runtime → Disconnect and delete runtime) and run Session B.")

## SESSION B — scoring (new session, T4 GPU). B1–B3 are Cells 1, 2 and 5 of the main notebook, unchanged

In [ ]:
# Cell B1 (= main Cell 1) — INSTALL (run once per session, then Runtime → Restart session)
import os, sys, subprocess
OPENVOICE_REPO, OPENVOICE_COMMIT = "/content/OpenVoice", "74a1d147b17a8c3092dd5430504bd83ef6c7eb23"
SEED_REPO, SEED_COMMIT = "/content/seed-vc", "51383efd921027683c89e5348211d93ff12ac2a8"
for repo, url, commit in [(OPENVOICE_REPO, "https://github.com/myshell-ai/OpenVoice.git", OPENVOICE_COMMIT),
                          (SEED_REPO, "https://github.com/Plachtaa/seed-vc.git", SEED_COMMIT)]:
    if not os.path.isdir(repo):
        subprocess.run(["git", "clone", url, repo], check=True)
    subprocess.run(["git", "checkout", commit], cwd=repo, check=True)
pkgs = [# OpenVoice (same as Untitled15, Step 3)
        "wavmark==0.0.3", "eng_to_ipa==0.0.2", "inflect==7.0.0", "unidecode==1.3.7", "pypinyin==0.50.0",
        "cn2an==0.5.22", "jieba==0.42.1", "langid==1.1.6",
        # Seed-VC (same as Untitled15, Step 7)
        "numpy==1.26.4", "scipy==1.13.1", "librosa==0.10.2", "transformers==4.46.3", "huggingface-hub>=0.28.1",
        "munch==4.0.0", "einops==0.8.0", "descript-audio-codec==1.0.0", "resemblyzer", "soundfile==0.12.1",
        "modelscope==1.18.1", "funasr==1.1.5", "hydra-core==1.3.2", "python-dotenv", "pyyaml",
        # scoring (same as Untitled13) + accelerate
        "speechbrain==1.0.3", "jiwer==3.0.3", "accelerate"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ Installed. NOW: Runtime → Restart session, then run Cell B2.")

In [ ]:
# Runtime configuration — edit these two lines consistently in all notebooks.
import os
DATA_ROOT = "/content/drive/MyDrive/IJEEE_VC_data"
RUN_DIR = "/content/drive/MyDrive/IJEEE_VC_data/reproduction_runs/review_run_01"
# DATA_ROOT contains audio/models. RUN_DIR is a new output folder, never results/.
os.environ["IJEEE_DATA_ROOT"] = DATA_ROOT
os.environ["IJEEE_RUN_DIR"] = RUN_DIR


In [ ]:
# Cell B2 (= main Cell 2) — PATHS AND SETTINGS
import os, sys, glob, json, time, random, shutil, tempfile, re, string, unicodedata, subprocess
os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["USE_FLAX"] = "0"
from google.colab import drive
drive.mount("/content/drive")
import numpy as np, pandas as pd, torch

BASE_DIR = os.environ.get("IJEEE_DATA_ROOT", "/content/drive/MyDrive/IJEEE_VC_data")
OUT = os.environ.get("IJEEE_RUN_DIR", os.path.join(BASE_DIR, "reproduction_runs", "review_run_01")); os.makedirs(OUT, exist_ok=True)

TUNING_FILES = [os.path.join(BASE_DIR, "selected_hybrid_data", f"joe_source_{i:02d}.wav") for i in (1, 2, 3)]
TUNING_TEXT = ["The monkey was planning to throw the net like the fishermen did.",
               "Gaze out to sea, picnic on benches and watch fishermen hook herring.",
               "We all have our own methods of dealing with money, but here are a couple of saving tricks that will grow your money."]
TARGET_REF = os.path.join(BASE_DIR, "selected_hybrid_data", "kathleen_reference.wav")
# EDIT if the two new conditioning references are stored elsewhere on Drive:
EXTRA_REFS = [os.path.join(BASE_DIR, "selected_hybrid_data", f"kathleen_conditioning_ref_{i:02d}.wav") for i in (2, 3)]
EVAL_CLIPS = sorted(glob.glob(os.path.join(BASE_DIR, "kathleen_independent_refs", "kathleen_eval_*.wav")))
JOE_ARCHIVE = os.path.join(BASE_DIR, "joe.tar.gz")
HOLDOUT10_META = os.path.join(BASE_DIR, "final_holdout_data", "holdout_metadata.csv")
HOLDOUT40_DIR = os.path.join(BASE_DIR, "final_holdout_data_40"); os.makedirs(HOLDOUT40_DIR, exist_ok=True)
HOLDOUT40_META = os.path.join(HOLDOUT40_DIR, "holdout40_metadata.csv")
EXISTING = os.path.join(BASE_DIR, "final_hybrid_comparison")
OV_CONVERTER = os.path.join(BASE_DIR, "models", "OpenVoiceV2", "converter")
N_HOLDOUT = 40
TUNING_IDS = {"0000000160", "0000000229", "0000000068"}

for p in TUNING_FILES + [TARGET_REF, JOE_ARCHIVE, HOLDOUT10_META, os.path.join(OV_CONVERTER, "checkpoint.pth")]:
    assert os.path.isfile(p), f"❌ missing: {p}"
for p in EXTRA_REFS:
    print("extra reference", os.path.basename(p), "found" if os.path.isfile(p) else "❌ NOT FOUND — fix the EDIT line")
print("Independent Kathleen evaluation clips:", len(EVAL_CLIPS))
assert len(EVAL_CLIPS) == 10, "❌ kathleen_eval_*.wav clips not found"
assert torch.cuda.is_available(), "❌ select a T4 GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def append_row(csv, row):
    pd.DataFrame([row]).to_csv(csv, mode="a", header=not os.path.exists(csv), index=False)
def done_keys(csv, cols):
    if not os.path.exists(csv): return set()
    return set(map(tuple, pd.read_csv(csv)[cols].astype(str).values))
print("✅ Paths ready")

In [ ]:
# Cell B3 (= main Cell 5) — SCORING MODELS (ECAPA and Whisper-small exactly as in Untitled13; plus WavLM, Whisper-large-v3, UTMOS)
import librosa, jiwer, torchaudio, logging
import torch.nn.functional as F
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
from speechbrain.inference.speaker import SpeakerRecognition
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, AutoFeatureExtractor, WavLMForXVector
logging.getLogger("speechbrain").setLevel(logging.WARNING)
DEV = "cuda"
speaker_model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                                savedir="/content/evaluation_models/ecapa_voxceleb", run_opts={"device": DEV})
ASR = {}
for name, mid, dtype in [("small", "openai/whisper-small", torch.float32), ("large", "openai/whisper-large-v3", torch.float16)]:
    proc = AutoProcessor.from_pretrained(mid)
    mdl = AutoModelForSpeechSeq2Seq.from_pretrained(mid, torch_dtype=dtype, use_safetensors=True).to(DEV).eval()
    mdl.config.forced_decoder_ids = None; mdl.generation_config.forced_decoder_ids = None
    ASR[name] = (proc, mdl, dtype)
wl_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
wl = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(DEV).eval()
utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(DEV).eval()

def load_audio_16k(p): return np.asarray(librosa.load(p, sr=16000, mono=True)[0], dtype=np.float32)
_EMB = {}
@torch.inference_mode()
def emb(p, kind):
    if (p, kind) in _EMB: return _EMB[(p, kind)]
    w = load_audio_16k(p)
    if kind == "ecapa":   # identical to Untitled13 extract_speaker_embedding
        e = speaker_model.encode_batch(torch.from_numpy(w).unsqueeze(0).to(DEV), wav_lens=torch.tensor([1.0]).to(DEV), normalize=False)
        e = F.normalize(e.squeeze().float(), p=2, dim=0).cpu()
    else:
        x = wl_fe(w, sampling_rate=16000, return_tensors="pt").to(DEV)
        e = F.normalize(wl(**x).embeddings.squeeze().float(), p=2, dim=0).cpu()
    _EMB[(p, kind)] = e
    return e
def cos(a, b, kind): return float(F.cosine_similarity(emb(a, kind), emb(b, kind), dim=0))

def join_spelled_letters(t):          # "N E U" -> "NEU" (reviewer fix), applied to reference and hypothesis
    return re.sub(r"\b(?:[A-Z] )+[A-Z]\b", lambda m: m.group(0).replace(" ", ""), str(t))
def normalize_transcript(text):        # identical to Untitled13 after the letter-joining step
    text = unicodedata.normalize("NFKC", join_spelled_letters(text)).lower()
    for a, b in [("’", "'"), ("–", " "), ("—", " "), ("-", " ")]: text = text.replace(a, b)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()
@torch.inference_mode()
def transcribe(p, which):
    proc, mdl, dtype = ASR[which]
    mi = proc(load_audio_16k(p), sampling_rate=16000, return_tensors="pt", return_attention_mask=True)
    args = dict(input_features=mi.input_features.to(DEV, dtype), language="en", task="transcribe", max_new_tokens=128)
    if getattr(mi, "attention_mask", None) is not None: args["attention_mask"] = mi.attention_mask.to(DEV)
    return proc.batch_decode(mdl.generate(**args), skip_special_tokens=True)[0].strip()
@torch.inference_mode()
def mos(p): return float(utmos(torch.from_numpy(load_audio_16k(p)).unsqueeze(0).to(DEV), 16000).item())

def score(out, text, src, cond_ref, eval_clips):
    r = dict(sim_ecapa_cond=cos(out, cond_ref, "ecapa"), sim_wavlm_cond=cos(out, cond_ref, "wavlm"),
             sim_ecapa_eval=float(np.mean([cos(out, c, "ecapa") for c in eval_clips])),
             sim_wavlm_eval=float(np.mean([cos(out, c, "wavlm") for c in eval_clips])),
             leak_ecapa_source=cos(out, src, "ecapa"), utmos=mos(out), duration_out=librosa.get_duration(path=out))
    ref_n = normalize_transcript(text)
    for which in ("small", "large"):
        hyp = transcribe(out, which); hyp_n = normalize_transcript(hyp)
        r[f"hyp_{which}"] = hyp; r[f"wer_{which}"] = jiwer.wer(ref_n, hyp_n); r[f"cer_{which}"] = jiwer.cer(ref_n, hyp_n)
        r[f"ref_words"] = len(ref_n.split())
    for k in list(_EMB):                 # keep only cached reference/eval/source embeddings
        if k[0] == out: _EMB.pop(k)
    return r
print("✅ Scoring models ready")

In [ ]:
# Cell B4 — SCORE THE 40 YOURTTS OUTPUTS + RECORD THE AUDIO FORMAT OF THE REFERENCES
import soundfile as sf
H40 = pd.read_csv(HOLDOUT40_META)
gen = pd.read_csv(os.path.join(OUT, "E15_yourtts_generation.csv")).sort_values("sample")
assert len(gen) == 40, "❌ run Session A first"
csv = os.path.join(OUT, "E15_yourtts_holdout40.csv")
done = set(pd.read_csv(csv)["sample"]) if os.path.exists(csv) else set()
for _, g in gen.iterrows():
    i = int(g["sample"])
    if i in done: continue
    r = H40[H40["sample"] == i].iloc[0]
    out = os.path.join(OUT, "E15_audio", g.output_file)
    s = score(out, r.transcript, r.prepared_wav, TARGET_REF, EVAL_CLIPS)
    row = dict(system=g.system, sample=i, tau=np.nan, steps=np.nan, src_duration=r.duration_seconds,
               t_openvoice=0.0, t_seedvc=0.0, latency=g.latency, latency_e2e=g.latency_e2e, rtf=g.rtf,
               t_asr=g.t_asr, t_tts=g.t_tts, asr_transcript=g.asr_transcript, **s)
    append_row(csv, row)
y = pd.read_csv(csv)
print(y[["sim_ecapa_cond", "sim_ecapa_eval", "sim_wavlm_eval", "leak_ecapa_source", "wer_large", "cer_large", "utmos", "latency", "rtf"]].mean().round(4))

# audio format of the files actually used (answers the open question on the Kathleen reference)
fmt = []
for p in [TARGET_REF] + EXTRA_REFS + EVAL_CLIPS[:2] + list(H40.prepared_wav[:2]):
    if os.path.isfile(p):
        i_ = sf.info(p); fmt.append(dict(file=os.path.basename(p), samplerate=i_.samplerate, channels=i_.channels,
                                        subtype=i_.subtype, format=i_.format, seconds=round(i_.duration, 3)))
fmt = pd.DataFrame(fmt); fmt.to_csv(os.path.join(OUT, "E15_reference_format.csv"), index=False); print(fmt)
print("✅ Scoring finished")

In [ ]:
# Cell B5 — PACK AND DOWNLOAD (CSV and JSON only; no audio)
import zipfile
from google.colab import files
z = "/content/IJEEE_yourtts_results.zip"
with zipfile.ZipFile(z, "w") as f:
    for n in ["E15_yourtts_generation.csv", "E15_yourtts_holdout40.csv", "E15_reference_format.csv", "environment_yourtts.json"]:
        p = os.path.join(OUT, n)
        if os.path.exists(p): f.write(p, n)
print(zipfile.ZipFile(z).namelist()); files.download(z)